# Chunking de manifestações longas
Compara duas configurações de `chunk_size` e `chunk_overlap` e visualiza os chunks em 2D.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

dados = pd.DataFrame(json.load(open('manifestacoes.json', encoding='utf-8')))
longas = dados[dados.texto.str.len() > 500].copy()
configuracoes = [(350, 60), (500, 100)]
modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
for tamanho, overlap in configuracoes:
    splitter = RecursiveCharacterTextSplitter(chunk_size=tamanho, chunk_overlap=overlap)
    chunks = [chunk for texto in longas.texto for chunk in splitter.split_text(texto)]
    vetores = modelo.encode(chunks, normalize_embeddings=True)
    pontos = PCA(n_components=2, random_state=42).fit_transform(vetores)
    plt.scatter(pontos[:,0], pontos[:,1]); plt.title(f'chunk_size={tamanho}, overlap={overlap}'); plt.show()
    print(f'Configuração {tamanho}/{overlap}: {len(chunks)} chunks')